In [1]:
import pandas as pd
import json
import logging
import hashlib

from pathlib import Path
from datetime import datetime, timezone

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
file_path = "salary_synthetic.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Dataset Shape:", df.shape)
print("Total Missing Values:", df.isna().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())

df.head()

Dataset loaded successfully.
Dataset Shape: (2000, 15)
Total Missing Values: 1200
Duplicate Rows: 0


,age,gender,education,experience_years,role_seniority,company_size,location_tier,skills_count,certifications,worked_remote,last_promotion_years_ago,salary_bdt,recent_project_description_length,survey_date,recent_note
0,33,Male,M.Sc,6.0,Senior,Enterprise,Tier-2,1.0,2.0,1,0.0,145185,57.0,2024-11-15,Worked on microservices deployment
1,29,Male,M.Sc,9.0,Junior,Enterprise,Remote,5.0,0.0,1,0.0,121262,50.0,2024-03-08,Built ML pipeline for preprocessing
2,34,Male,B.Sc+Cert,NaN,Lead,Enterprise,Remote,5.0,1.0,1,5.0,184875,50.0,2024-09-01,Experience in data cleaning and ETL
3,39,Male,B.Sc,16.0,Mid,Enterprise,Tier-1,5.0,0.0,1,3.0,180105,59.0,2024-01-26,Contributed to open-source NLP repo
4,29,Female,B.Sc,8.0,Junior,SME,Tier-1,5.0,2.0,0,7.0,114750,NaN,2023-12-08,Built ML pipeline for preprocessing


In [3]:
df_original = df.copy()

print("Original dataset backup created.")

Original dataset backup created.


In [4]:
# Automatically infer suitable datatypes
df = df.infer_objects()

# Convert survey_date into datetime
df["survey_date"] = pd.to_datetime(
    df["survey_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

print("survey_date datatype:", df["survey_date"].dtype)
print("Invalid dates:", df["survey_date"].isna().sum())

survey_date datatype: datetime64[us]
Invalid dates: 0


In [5]:
categorical_columns = [
    "gender",
    "education",
    "role_seniority",
    "company_size",
    "location_tier",
    "recent_note"
]

for column in categorical_columns:
    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print("Categorical values standardized successfully.")

Categorical values standardized successfully.


In [6]:
df["survey_year"] = df["survey_date"].dt.year
df["survey_month"] = df["survey_date"].dt.month
df["survey_day"] = df["survey_date"].dt.day

print("Date features created.")

df[
    [
        "survey_date",
        "survey_year",
        "survey_month",
        "survey_day"
    ]
].head()

Date features created.


,survey_date,survey_year,survey_month,survey_day
0,2024-11-15,2024,11,15
1,2024-03-08,2024,3,8
2,2024-09-01,2024,9,1
3,2024-01-26,2024,1,26
4,2023-12-08,2023,12,8


In [7]:
DATA_VERSION = "v1.1"

df.insert(
    0,
    "data_version",
    DATA_VERSION
)

print("Dataset Version:", DATA_VERSION)
print("Updated Shape:", df.shape)

Dataset Version: v1.1
Updated Shape: (2000, 19)


In [8]:
output_folder = Path("task8_outputs")

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

print("Output folder:", output_folder.resolve())

Output folder: C:\Users\DELL\OneDrive\New folder\Desktop\brainybeam\21-08-26\task8_outputs


In [9]:
export_df = df.copy()

export_df["survey_date"] = (
    export_df["survey_date"]
    .dt.strftime("%Y-%m-%d")
)

print("Dataset prepared for export.")
print("Export Shape:", export_df.shape)

Dataset prepared for export.
Export Shape: (2000, 19)


In [10]:
csv_path = output_folder / "salary_cleaned_preimputation_v1_1.csv"

parquet_path = (
    output_folder /
    "salary_cleaned_preimputation_v1_1.parquet"
)

json_path = output_folder / "salary_cleaned_preimputation_v1_1.json"

log_path = output_folder / "preprocessing_v1_1.log"

manifest_path = output_folder / "manifest_v1_1.json"

print("Versioned filenames created.")

Versioned filenames created.


In [12]:
export_df.to_csv(
    csv_path,
    index=False
)

print("CSV file saved successfully:")
print(csv_path)

CSV file saved successfully:
task8_outputs\salary_cleaned_preimputation_v1_1.csv


In [13]:
export_df.to_parquet(
    parquet_path,
    index=False,
    engine="pyarrow"
)

print("Parquet file saved successfully:")
print(parquet_path)

Parquet file saved successfully:
task8_outputs\salary_cleaned_preimputation_v1_1.parquet


In [14]:
export_df.to_json(
    json_path,
    orient="records",
    lines=True,
    date_format="iso"
)

print("JSON file saved successfully:")
print(json_path)

JSON file saved successfully:
task8_outputs\salary_cleaned_preimputation_v1_1.json


In [15]:
logger = logging.getLogger("SalaryInsight_Preprocessing")

logger.setLevel(logging.INFO)

# Notebook mein duplicate log entries avoid karega
logger.handlers.clear()

file_handler = logging.FileHandler(
    log_path,
    mode="w",
    encoding="utf-8"
)

log_format = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s"
)

file_handler.setFormatter(log_format)

logger.addHandler(file_handler)

print("Data logging configured.")

Data logging configured.


In [16]:
logger.info("SalaryInsight preprocessing started")

logger.info("Source file: %s", file_path)

logger.info(
    "Original dataset shape: %s",
    df_original.shape
)

logger.info(
    "Original missing values: %d",
    df_original.isna().sum().sum()
)

logger.info(
    "Original duplicate rows: %d",
    df_original.duplicated().sum()
)

logger.info(
    "Applied Pandas infer_objects()"
)

logger.info(
    "Converted survey_date into datetime"
)

logger.info(
    "Standardized categorical values"
)

logger.info(
    "Created survey_year, survey_month and survey_day"
)

logger.info(
    "Assigned dataset version: %s",
    DATA_VERSION
)

logger.info(
    "Exported dataset in CSV, Parquet and JSON formats"
)

logger.info(
    "Missing values preserved for Task 9: %d",
    export_df.isna().sum().sum()
)

logger.info("SalaryInsight preprocessing completed")

print("Preprocessing steps recorded successfully.")

Preprocessing steps recorded successfully.


In [17]:
for handler in logger.handlers:
    handler.flush()

with open(log_path, "r", encoding="utf-8") as log_file:
    print(log_file.read())

2026-08-23 21:49:03,721 | INFO | SalaryInsight preprocessing started
2026-08-23 21:49:03,722 | INFO | Source file: salary_synthetic.csv
2026-08-23 21:49:03,722 | INFO | Original dataset shape: (2000, 15)
2026-08-23 21:49:03,727 | INFO | Original missing values: 1200
2026-08-23 21:49:03,741 | INFO | Original duplicate rows: 0
2026-08-23 21:49:03,743 | INFO | Applied Pandas infer_objects()
2026-08-23 21:49:03,744 | INFO | Converted survey_date into datetime
2026-08-23 21:49:03,744 | INFO | Standardized categorical values
2026-08-23 21:49:03,745 | INFO | Created survey_year, survey_month and survey_day
2026-08-23 21:49:03,746 | INFO | Assigned dataset version: v1.1
2026-08-23 21:49:03,747 | INFO | Exported dataset in CSV, Parquet and JSON formats
2026-08-23 21:49:03,749 | INFO | Missing values preserved for Task 9: 1200
2026-08-23 21:49:03,750 | INFO | SalaryInsight preprocessing completed



In [18]:
def calculate_sha256(file_name):
    
    sha256_hash = hashlib.sha256()
    
    with open(file_name, "rb") as file:
        
        for block in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            sha256_hash.update(block)
    
    return sha256_hash.hexdigest()

In [19]:
processing_time = datetime.now(
    timezone.utc
).replace(microsecond=0).isoformat()

manifest = {
    "project": "SalaryInsight",
    "dataset_version": DATA_VERSION,
    "dataset_status": "pre-imputation",
    "created_at_utc": processing_time,
    "source_file": file_path,
    "original_rows": int(df_original.shape[0]),
    "original_columns": int(df_original.shape[1]),
    "exported_rows": int(export_df.shape[0]),
    "exported_columns": int(export_df.shape[1]),
    "target_column": "salary_bdt",
    "remaining_missing_values": int(
        export_df.isna().sum().sum()
    ),
    "duplicate_rows": int(
        export_df.duplicated().sum()
    ),
    "files": {
        "csv": {
            "filename": csv_path.name,
            "sha256": calculate_sha256(csv_path)
        },
        "parquet": {
            "filename": parquet_path.name,
            "sha256": calculate_sha256(parquet_path)
        },
        "json": {
            "filename": json_path.name,
            "sha256": calculate_sha256(json_path)
        },
        "log": {
            "filename": log_path.name,
            "sha256": calculate_sha256(log_path)
        }
    }
}

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as manifest_file:
    
    json.dump(
        manifest,
        manifest_file,
        indent=4
    )

print("Version manifest created successfully:")
print(manifest_path)

Version manifest created successfully:
task8_outputs\manifest_v1_1.json


In [20]:
csv_check = pd.read_csv(csv_path)

parquet_check = pd.read_parquet(
    parquet_path,
    engine="pyarrow"
)

json_check = pd.read_json(
    json_path,
    orient="records",
    lines=True
)

print("All exported files loaded successfully.")

All exported files loaded successfully.


In [21]:
verification = pd.DataFrame({
    "Format": [
        "Original Export Data",
        "CSV",
        "Parquet",
        "JSON"
    ],
    "Rows": [
        export_df.shape[0],
        csv_check.shape[0],
        parquet_check.shape[0],
        json_check.shape[0]
    ],
    "Columns": [
        export_df.shape[1],
        csv_check.shape[1],
        parquet_check.shape[1],
        json_check.shape[1]
    ],
    "Missing Values": [
        export_df.isna().sum().sum(),
        csv_check.isna().sum().sum(),
        parquet_check.isna().sum().sum(),
        json_check.isna().sum().sum()
    ]
})

verification

,Format,Rows,Columns,Missing Values
0,Original Export Data,2000,19,1200
1,CSV,2000,19,1200
2,Parquet,2000,19,1200
3,JSON,2000,19,1200


In [22]:
all_shapes_match = (
    export_df.shape
    == csv_check.shape
    == parquet_check.shape
    == json_check.shape
)

all_missing_counts_match = (
    export_df.isna().sum().sum()
    == csv_check.isna().sum().sum()
    == parquet_check.isna().sum().sum()
    == json_check.isna().sum().sum()
)

print("All shapes match:", all_shapes_match)

print(
    "All missing-value counts match:",
    all_missing_counts_match
)

All shapes match: True
All missing-value counts match: True


In [25]:
for file in output_folder.iterdir():
    print(
        file.name,
        "—",
        file.stat().st_size,
        "bytes"
    )

manifest_v1_1.json — 1182 bytes
preprocessing_v1_1.log — 913 bytes
salary_cleaned_preimputation_v1_1.csv — 238387 bytes
salary_cleaned_preimputation_v1_1.json — 882121 bytes
salary_cleaned_preimputation_v1_1.parquet — 44741 bytes
